<a href="https://colab.research.google.com/github/cs-iuu/ai-2026-fall-nlp/blob/main/notebook/15_Static_Contextual_Embeddings_Guided_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guided Lab: Static and Contextual Embeddings

This notebook is a **guided demonstration**, not the solution to the TutorFlow programming assignment.

It intentionally uses:

- pre-tokenized text instead of implementing a tokenizer;
- `gensim` instead of manually building context-count vectors;
- library similarity functions instead of implementing cosine similarity;
- a pretrained multilingual Transformer instead of implementing self-attention.

By the end of the lab, you should be able to explain:

1. Why Word2Vec assigns one static vector to each word type.
2. Why a Transformer produces a different vector for each occurrence.
3. Why contextual vectors are useful for word-sense disambiguation.


## 0. Setup

Run this notebook in Google Colab. A GPU is helpful for the Transformer section but is not required for this small example.


In [ ]:
%pip -q install gensim transformers sentencepiece


In [ ]:
import random
import numpy as np
import torch

from gensim.models import Word2Vec
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# Part I — Static embeddings with Word2Vec

The corpus below is already tokenized so that this lab does not reveal the tokenization implementation required by the TutorFlow assignment.

The repeated patterns help a very small teaching corpus produce visible relationships. The resulting vectors are demonstrations—not reliable linguistic models.


In [ ]:
tokenized_corpus = [
    ["муур", "гэрийн", "тэжээвэр", "амьтан"],
    ["нохой", "гэрийн", "тэжээвэр", "амьтан"],
    ["муур", "сүү", "ууж", "байна"],
    ["нохой", "ус", "ууж", "байна"],
    ["хонь", "бэлчээрт", "идээшилж", "байна"],
    ["ямаа", "бэлчээрт", "идээшилж", "байна"],
    ["морь", "талд", "давхиж", "байна"],
    ["тэмээ", "говьд", "явж", "байна"],
    ["оюутан", "ном", "уншиж", "байна"],
    ["сурагч", "ном", "уншиж", "байна"],
    ["багш", "хичээл", "зааж", "байна"],
    ["оюутан", "хичээл", "сурч", "байна"],
    ["миний", "толгой", "өвдөж", "байна"],
    ["эмч", "толгой", "шалгаж", "байна"],
    ["уулын", "толгой", "цастай", "байна"],
    ["уулын", "орой", "цастай", "байна"],
    ["таван", "толгой", "мал", "байна"],
    ["зуун", "толгой", "хонь", "байна"],
] * 10

print("Training sentences:", len(tokenized_corpus))
print("Example:", tokenized_corpus[0])


## 1. Train CBOW and Skip-gram

- CBOW predicts the center word from surrounding words.
- Skip-gram predicts surrounding words from the center word.

Both produce one stored vector per vocabulary word.


In [ ]:
settings = dict(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    seed=SEED,
    epochs=100,
)

cbow_model = Word2Vec(sg=0, **settings)
skipgram_model = Word2Vec(sg=1, **settings)

print("Vocabulary size:", len(skipgram_model.wv))
print("Vector dimensions:", skipgram_model.wv.vector_size)


## 2. Use built-in similarity operations

These library calls deliberately avoid showing the manual cosine-similarity solution required in TutorFlow.


In [ ]:
query_words = ["муур", "хонь", "оюутан", "толгой"]

for word in query_words:
    print(f"\n{word} — CBOW")
    print(cbow_model.wv.most_similar(word, topn=4))
    print(f"{word} — Skip-gram")
    print(skipgram_model.wv.most_similar(word, topn=4))


In [ ]:
comparison_pairs = [
    ("муур", "нохой"),
    ("хонь", "ямаа"),
    ("оюутан", "сурагч"),
    ("уулын", "орой"),
    ("муур", "ном"),
]

for first, second in comparison_pairs:
    cbow_score = cbow_model.wv.similarity(first, second)
    skipgram_score = skipgram_model.wv.similarity(first, second)
    print(f"{first:<10} {second:<10} CBOW={cbow_score:.3f}  Skip-gram={skipgram_score:.3f}")


### Checkpoint A

Discuss with a partner:

1. Which related words received relatively high similarity?
2. Which results appear unreasonable?
3. How could corpus size and repeated sentence patterns influence the results?

**TODO:** Record one reasonable and one unreasonable result.


## 3. Demonstrate the static polysemy limitation

Word2Vec retrieves the same vector for the exact token `толгой`, regardless of the sentence in which it occurs.


In [ ]:
static_examples = [
    "Миний толгой өвдөж байна.",
    "Уулын толгой цастай байна.",
    "Зуун толгой хонь байна.",
]

static_vector = skipgram_model.wv["толгой"]

for sentence in static_examples:
    print(sentence)
    print("First five vector values:", np.round(static_vector[:5], 4))


The vector is multidimensional and reflects all training contexts, but it is still **one blended word-type vector**. It does not become a body-part vector in one sentence and a livestock-counting vector in another.


# Part II — Contextual embeddings with XLM-RoBERTa

A contextual model processes the complete sentence and generates a vector for each token occurrence. Therefore, the three uses of `толгой` can receive three different vectors.


In [ ]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Loaded:", MODEL_NAME)


## 4. Extract the contextual vector of the target occurrence

The function below uses character offsets to locate the model tokens that overlap the target word. This is provided as infrastructure for the lab; students are not expected to reproduce it in TutorFlow.


In [ ]:
@torch.no_grad()
def target_embedding(sentence, target_form):
    target_start = sentence.lower().find(target_form.lower())
    if target_start < 0:
        raise ValueError(f"Target {target_form!r} was not found")
    target_end = target_start + len(target_form)

    encoded = tokenizer(
        sentence,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
    )
    offsets = encoded.pop("offset_mapping")[0].tolist()
    inputs = {name: tensor.to(device) for name, tensor in encoded.items()}
    hidden = model(**inputs).last_hidden_state[0]

    indices = [
        i for i, (start, end) in enumerate(offsets)
        if end > start and start < target_end and end > target_start
    ]
    if not indices:
        raise ValueError("No model token overlaps the target")

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0, indices].tolist())
    vector = hidden[indices].mean(dim=0).cpu()
    return vector, tokens


In [ ]:
contextual_examples = [
    {"sentence": "Миний толгой өвдөж байна.", "target": "толгой", "sense": "body part"},
    {"sentence": "Уулын толгой цастай байна.", "target": "толгой", "sense": "top/summit"},
    {"sentence": "Зуун толгой хонь байна.", "target": "толгой", "sense": "livestock unit"},
    {"sentence": "Эмч миний толгойг шалгав.", "target": "толгойг", "sense": "body part"},
    {"sentence": "Өндөр уулын орой цастай байна.", "target": "орой", "sense": "summit comparison"},
]

contextual_vectors = []

for item in contextual_examples:
    vector, tokens = target_embedding(item["sentence"], item["target"])
    contextual_vectors.append(vector)
    print(item["sense"], "|", item["target"], "->", tokens)


## 5. Compare occurrence vectors

The following uses PyTorch's built-in normalization and matrix multiplication. It does not reveal the manual cosine implementation from the TutorFlow assignment.


In [ ]:
matrix = torch.stack(contextual_vectors)
normalized = torch.nn.functional.normalize(matrix, p=2, dim=1)
similarity_matrix = normalized @ normalized.T

labels = [item["sense"] for item in contextual_examples]

print("Contextual target-word cosine similarities:\n")
print(" " * 20 + " ".join(f"{i:>7}" for i in range(len(labels))))
for i, row in enumerate(similarity_matrix):
    values = " ".join(f"{value.item():7.3f}" for value in row)
    print(f"{i}: {labels[i]:<16} {values}")


### Checkpoint B

Inspect the contextual similarity matrix.

1. Is the body-part use closer to the other body-part use than to the livestock use?
2. Is the summit use related to the separate word `орой`?
3. Are all similarities sensible? Why might a multilingual model make mistakes on Mongolian?

**TODO:** Record two observations.


# Part III — Static versus contextual

| Property | Word2Vec | XLM-R contextual embedding |
|---|---|---|
| Unit represented | Word type | Token occurrence |
| Same word in different sentences | Same vector | Usually different vectors |
| Uses surrounding context | During training | During every prediction |
| Handles polysemy directly | No | Better, but not perfectly |
| Computational cost | Relatively low | Relatively high |


## 6. Student experiment

Choose one Mongolian polysemous noun other than `толгой`.

1. Write at least six sentences: at least three examples for each of two senses.
2. Add the tokenized sentences to the Word2Vec corpus and retrain.
3. Add the original sentences and exact target forms to `contextual_examples`.
4. Compare the Word2Vec word-type vector with the contextual occurrence similarities.
5. Write 150–250 words explaining which representation separates the senses better.

Do not claim that six sentences are sufficient for reliable model evaluation. This is a small demonstration experiment.


## Submission checklist

- [ ] Chose one additional polysemous Mongolian noun
- [ ] Added at least six checked sentences
- [ ] Ran both Word2Vec and XLM-R sections
- [ ] Reported at least two similarity observations
- [ ] Explained the static polysemy limitation
- [ ] Wrote a 150–250-word comparison
- [ ] Disclosed any AI assistance
